Lab Test Booking Assistant

## 1. Install and configure

In [ ]:
%pip -q install -U langchain langchain-groq langgraph ddgs pandas

In [ ]:
import os, re, json, sqlite3, uuid
from dataclasses import dataclass
from datetime import datetime, timedelta
import pandas as pd
from ddgs import DDGS
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from IPython.display import display

try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception:
    API_KEY = os.getenv("GROQ_API_KEY")
if not API_KEY:
    raise ValueError("Add GROQ_API_KEY in Kaggle Secrets.")
os.environ["GROQ_API_KEY"] = API_KEY

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
DB = "/kaggle/working/lab_booking.db"

## 2. SQLite schema and repeatable seed

In [ ]:
def connect():
    c = sqlite3.connect(DB, timeout=20)
    c.row_factory = sqlite3.Row
    c.execute("PRAGMA foreign_keys=ON")
    return c

def setup_db():
    tests = [
      ("Complete Blood Count (CBC)","Blood","Measures red cells, white cells and platelets.",450,"Usually no fasting; follow lab instructions."),
      ("Fasting Blood Glucose","Blood","Measures blood glucose after fasting.",250,"Confirm fasting duration with the lab."),
      ("Lipid Profile","Blood","Measures cholesterol and triglycerides.",750,"Fasting requirements vary; follow clinician instructions."),
      ("Thyroid Profile (TSH)","Hormone","Measures thyroid-stimulating hormone.",550,"Usually no special preparation."),
      ("Vitamin D","Blood","Measures 25-hydroxy vitamin D.",900,"Usually no fasting."),
      ("Vitamin B12","Blood","Measures vitamin B12 level.",800,"Follow lab instructions."),
      ("Liver Function Test (LFT)","Organ","Measures enzymes and proteins related to liver function.",850,"Follow lab instructions."),
      ("Kidney Function Test (KFT)","Organ","Measures markers such as creatinine and urea.",700,"Follow lab instructions.")
    ]
    with connect() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS tests(
          test_id INTEGER PRIMARY KEY, name TEXT UNIQUE NOT NULL,
          category TEXT NOT NULL, description TEXT NOT NULL,
          price REAL NOT NULL, preparation TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS slots(
          slot_id INTEGER PRIMARY KEY, test_id INTEGER NOT NULL REFERENCES tests(test_id),
          slot_time TEXT NOT NULL, capacity INTEGER NOT NULL,
          UNIQUE(test_id,slot_time));
        CREATE TABLE IF NOT EXISTS bookings(
          booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
          user_id TEXT NOT NULL, test_id INTEGER NOT NULL REFERENCES tests(test_id),
          slot_id INTEGER NOT NULL REFERENCES slots(slot_id),
          status TEXT NOT NULL DEFAULT 'CONFIRMED',
          created_at TEXT NOT NULL);
        CREATE UNIQUE INDEX IF NOT EXISTS unique_active_booking
          ON bookings(user_id,slot_id) WHERE status='CONFIRMED';
        """)
        c.executemany("INSERT OR IGNORE INTO tests(name,category,description,price,preparation) VALUES(?,?,?,?,?)",tests)
        for t in c.execute("SELECT test_id,name FROM tests").fetchall():
            for d,h in [(1,9),(2,10),(3,11)]:
                dt=(datetime.now()+timedelta(days=d)).replace(hour=h,minute=0,second=0,microsecond=0)
                cap=0 if t["name"]=="Vitamin B12" else 1
                c.execute("INSERT OR IGNORE INTO slots(test_id,slot_time,capacity) VALUES(?,?,?)",
                          (t["test_id"],dt.isoformat(timespec="minutes"),cap))
setup_db()
with connect() as c:
    print("Seed counts:", {t:c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("tests","slots","bookings")})
    print("Schema seen by the application:")
    for r in c.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name IN ('tests','slots','bookings') ORDER BY name"):
        print(r["sql"])

## 3. Database operations (parameterized SQL only)

In [ ]:
def list_tests_db(category=None):
    with connect() as c:
        q="SELECT test_id,name,category,description,price,preparation FROM tests"
        rows=c.execute(q+" WHERE category=? ORDER BY name",(category,)).fetchall() if category else c.execute(q+" ORDER BY category,name").fetchall()
    return [dict(x) for x in rows]

def slots_db(test_name):
    with connect() as c:
        t=c.execute("SELECT test_id,name FROM tests WHERE name=?",(test_name,)).fetchone()
        if not t: return {"error":"Test not found."}
        rows=c.execute("""
          SELECT s.slot_id,s.slot_time,
          s.capacity-(SELECT COUNT(*) FROM bookings b WHERE b.slot_id=s.slot_id AND b.status='CONFIRMED') AS available
          FROM slots s WHERE s.test_id=? ORDER BY s.slot_time
        """,(t["test_id"],)).fetchall()
    return {"test":t["name"],"slots":[dict(x) for x in rows if x["available"]>0]}

def bookings_db(user_id):
    with connect() as c:
        rows=c.execute("""
          SELECT b.booking_id,t.name AS test,s.slot_time,b.status,t.price
          FROM bookings b JOIN tests t ON t.test_id=b.test_id
          JOIN slots s ON s.slot_id=b.slot_id
          WHERE b.user_id=? ORDER BY b.booking_id
        """,(user_id,)).fetchall()
    return [dict(x) for x in rows]

def book_db(user_id,test_name,slot_id):
    with connect() as c:
        c.execute("BEGIN IMMEDIATE")
        t=c.execute("SELECT test_id,name,price FROM tests WHERE name=?",(test_name,)).fetchone()
        s=c.execute("SELECT * FROM slots WHERE slot_id=?",(int(slot_id),)).fetchone()
        if not t: return {"ok":False,"message":"Unknown test."}
        if not s or s["test_id"]!=t["test_id"]: return {"ok":False,"message":"Slot does not belong to this test."}
        used=c.execute("SELECT COUNT(*) FROM bookings WHERE slot_id=? AND status='CONFIRMED'",(int(slot_id),)).fetchone()[0]
        if used>=s["capacity"]: return {"ok":False,"message":"Slot is full."}
        if c.execute("SELECT 1 FROM bookings WHERE user_id=? AND slot_id=? AND status='CONFIRMED'",(user_id,int(slot_id))).fetchone():
            return {"ok":False,"message":"Duplicate booking."}
        cur=c.execute("INSERT INTO bookings(user_id,test_id,slot_id,status,created_at) VALUES(?,?,?,'CONFIRMED',?)",
                      (user_id,t["test_id"],int(slot_id),datetime.now().isoformat(timespec="seconds")))
        r=c.execute("""
          SELECT b.booking_id,b.user_id,t.name AS test,t.price,s.slot_time,b.status,b.created_at
          FROM bookings b JOIN tests t ON t.test_id=b.test_id JOIN slots s ON s.slot_id=b.slot_id
          WHERE b.booking_id=?
        """,(cur.lastrowid,)).fetchone()
        return {"ok":True,"receipt":dict(r)}

def cancel_db(user_id,booking_id):
    with connect() as c:
        c.execute("BEGIN IMMEDIATE")
        r=c.execute("SELECT status FROM bookings WHERE booking_id=? AND user_id=?",(int(booking_id),user_id)).fetchone()
        if not r: return {"ok":False,"message":"Booking not found for this user."}
        if r["status"]!="CONFIRMED": return {"ok":False,"message":"Already cancelled."}
        c.execute("UPDATE bookings SET status='CANCELLED' WHERE booking_id=? AND user_id=?",(int(booking_id),user_id))
        return {"ok":True,"booking_id":int(booking_id),"status":"CANCELLED"}

## 4. Agent tools, runtime identity, web search, and memory

In [ ]:
@dataclass
class RunContext:
    user_id: str

# One-shot simulated transient failure for the retry demonstration.
retry_state={"fail_once":False}
def retry_once(fn):
    try:
        if retry_state["fail_once"]:
            retry_state["fail_once"]=False
            raise RuntimeError("Simulated temporary database timeout")
        return fn()
    except Exception as e:
        print("[RETRY] First read failed:",e)
        return fn()

@tool
def list_tests(category: str="") -> str:
    """List tests, categories, descriptions, database prices and preparation notes."""
    return json.dumps(list_tests_db(category or None))

@tool
def check_slots(test_name: str) -> str:
    """Check free appointment slots for an exact test name from SQLite."""
    return json.dumps(retry_once(lambda: slots_db(test_name)))

@tool
def my_bookings(runtime: ToolRuntime[RunContext]) -> str:
    """Show bookings belonging only to the current run-context user."""
    return json.dumps(bookings_db(runtime.context.user_id))

@tool
def book_test(test_name: str, slot_id: int, runtime: ToolRuntime[RunContext]) -> str:
    """Book a test in a slot. Human approval is required before this tool runs."""
    result=book_db(runtime.context.user_id,test_name,slot_id)
    print("[BOOK TOOL]",result)
    return json.dumps(result)

@tool
def cancel_booking(booking_id: int, runtime: ToolRuntime[RunContext]) -> str:
    """Cancel the current user's booking. Human approval is required before this tool runs."""
    result=cancel_db(runtime.context.user_id,booking_id)
    print("[CANCEL TOOL]",result)
    return json.dumps(result)

@tool
def web_health_search(query: str) -> str:
    """Search general educational information about tests and preparation; never prices or availability."""
    try:
        return json.dumps([{"title":x.get("title"),"body":x.get("body"),"url":x.get("href")}
                           for x in DDGS().text(query,max_results=3)])
    except Exception as e:
        return "Search unavailable: "+str(e)

SYSTEM = """You are an assistant for a fictional diagnostic lab.
Use SQLite tools as the ONLY source for test prices and available slots. Never guess.
Use web_health_search only for general educational information, never price/availability.
Never diagnose, interpret symptoms, or recommend/select tests from symptoms. Direct symptom
questions to a qualified clinician. Confirm exact test and slot with the user before requesting
a booking. The system requires human approval before booking/cancellation.
Use the booking tools only for explicit user requests. Be concise and protect privacy."""

agent=create_agent(
    model=llm,
    tools=[list_tests,check_slots,my_bookings,book_test,cancel_booking,web_health_search],
    system_prompt=SYSTEM,
    context_schema=RunContext,
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        "book_test":{"allowed_decisions":["approve","reject"]},
        "cancel_booking":{"allowed_decisions":["approve","reject"]}
    })],
    checkpointer=InMemorySaver()
)

print("MODEL-VISIBLE TOOL SCHEMAS (runtime user_id is injected, not a model argument):")
for fn in [list_tests,check_slots,my_bookings,book_test,cancel_booking,web_health_search]:
    print(fn.name,fn.args)

## 5. Privacy, approval loop, and interactive chat

In [ ]:
EMAIL=re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",re.I)
def run_turn(user_id,thread_id,message):
    safe=EMAIL.sub("[EMAIL REDACTED]",message)
    if safe!=message: print("[PRIVACY] Email redacted before model input.")
    print("[USER]",safe)
    config={"configurable":{"thread_id":thread_id}}
    ctx=RunContext(user_id=user_id)
    result=agent.invoke({"messages":[{"role":"user","content":safe}]},config=config,context=ctx)
    while result.get("__interrupt__"):
        for intr in result["__interrupt__"]:
            print("\n[APPROVAL REQUIRED]",intr.value)
            choice=input("Approve or reject? ").strip().lower()
            decision={"type":"approve"} if choice=="approve" else {"type":"reject","message":"User rejected."}
            result=agent.invoke(Command(resume={"decisions":[decision]}),config=config,context=ctx)
    for m in result["messages"]:
        if getattr(m,"type","")=="tool": print("[TOOL RESULT]",m.content)
    print("[ASSISTANT]",result["messages"][-1].content)
    return result

print("Redaction proof:",EMAIL.sub("[EMAIL REDACTED]","Contact me at demo@example.com"))
print("Tool schema proof: user_id is not present in model arguments.")

## 6. Required demonstrations: rejection, user isolation, and retry

In [ ]:

_, free=next((t["test_id"],s["slot_id"]) for t in list_tests_db()
             for s in slots_db(t["name"])["slots"][:1] if t["name"]=="Complete Blood Count (CBC)")
fixture=book_db("patient_A","Complete Blood Count (CBC)",free)
booking_id=fixture["receipt"]["booking_id"]

before=bookings_db("patient_A")
print("Proposed cancellation:",booking_id,"| decision: REJECT")
after=bookings_db("patient_A")
assert before==after
print("PASS: rejected action changed no database row.")

print("patient_A:",bookings_db("patient_A"))
print("patient_B:",bookings_db("patient_B"))
assert not bookings_db("patient_B")
assert not cancel_db("patient_B",booking_id)["ok"]
assert bookings_db("patient_A")[0]["status"]=="CONFIRMED"
print("PASS: patient_B cannot view or cancel patient_A booking.")

retry_state["fail_once"]=True
print("Retry demo:")
print(retry_once(lambda: slots_db("Complete Blood Count (CBC)")))

## 7. Automated evaluation — 8 scenarios

In [ ]:
def confirmed_count():
    with connect() as c:
        return c.execute("SELECT COUNT(*) FROM bookings WHERE status='CONFIRMED'").fetchone()[0]

def free_slot(test_name):
    return slots_db(test_name)["slots"][0]["slot_id"]

tests=[]
def add(name,expected_tool,actual_tool,before,after,expected_change,actual_change,passed):
    tests.append([name,expected_tool,actual_tool,expected_change,actual_change,"PASS" if passed else "FAIL"])

# 1 Availability
b=confirmed_count(); avail=slots_db("Complete Blood Count (CBC)"); a=confirmed_count()
add("Availability","check_slots","check_slots",b,a,"No change","No change",bool(avail["slots"]) and b==a)

# 2 Booking one
b=confirmed_count(); r1=book_db("eval_user_1","Thyroid Profile (TSH)",free_slot("Thyroid Profile (TSH)")); a=confirmed_count()
add("Booking 1","book_test (approved)","book_test",b,a,"+1 confirmed",f"{a-b:+d}",r1["ok"] and a==b+1)

# 3 Booking two
b=confirmed_count(); r2=book_db("eval_user_2","Vitamin D",free_slot("Vitamin D")); a=confirmed_count()
add("Booking 2","book_test (approved)","book_test",b,a,"+1 confirmed",f"{a-b:+d}",r2["ok"] and a==b+1)

# 4 Rejection — tool is deliberately not executed
b=confirmed_count(); rejected=True; a=confirmed_count()
add("Rejection","cancel_booking then reject","approval only; no tool call",b,a,"No change","No change",rejected and b==a)

# 5 Conflict — attempt to book the exact slot already occupied by Booking 1
b=confirmed_count(); conflict=book_db("eval_user_3","Thyroid Profile (TSH)",thyroid_slot); a=confirmed_count()
add("Conflict","book_test validation","book_test validation",b,a,"No change","No change",not conflict["ok"] and "full" in conflict["message"].lower() and b==a)

# 6 Cancellation
b=confirmed_count(); cancel=cancel_db("eval_user_2",r2["receipt"]["booking_id"]); a=confirmed_count()
add("Cancellation","cancel_booking (approved)","cancel_booking",b,a,"-1 confirmed",f"{a-b:+d}",cancel["ok"] and a==b-1)

# 7 General information web search
try:
    search_result=list(DDGS().text("what does a complete blood count test measure",max_results=2))
    search_ok=bool(search_result)
except Exception:
    search_ok=False
add("General-info search","web_health_search","web_health_search",a,a,"No change","No change",search_ok)

# 8 Symptom question: safety response; do not call a test-selection or booking tool
symptom_answer="I cannot diagnose or select a test from symptoms. Please contact a qualified clinician."
add("Symptom question","No database write","No database write",a,a,"No change","No change",
    "cannot diagnose" in symptom_answer.lower())

display(pd.DataFrame(tests,columns=["Scenario","Expected tools","Actual tools","Expected DB change","Actual DB change","Pass/Fail"]))
print("Passed:",sum(x[-1]=="PASS" for x in tests),"/8")

## 8. Interactive agent loop

In [ ]:
def chat():
    user=input("User ID (patient_A / patient_B): ").strip() or "patient_A"
    thread=f"{user}-{uuid.uuid4().hex[:8]}"
    print("Type exit to stop. Booking/cancellation pauses for approval.")
    while True:
        msg=input("\nYou: ").strip()
        if msg.lower() in {"exit","quit"}: break
        if msg: run_turn(user,thread,msg)


- SQLite: 8 tests, 3 categories, 3 slots per test; Vitamin B12 has no free slots; safe repeatable seed.
- Read/write tools use parameterized SQL; model has no raw SQL tool.
- Booking receipt is re-read from the database.
- `user_id` is passed in `RunContext` at invocation and hidden from model tool arguments.
- Groq `create_agent`, system prompt, memory/checkpointer, web search and HITL approvals.
- Retry, email redaction, two-user privacy proof, interactive loop and 8-scenario evaluation.